# Lab 4: How Uncertain Is the Forecast, and How Does Skill Change With the Horizon? (JPM)

**Owner:** Member 4 · **Marking items covered:** 5. Uncertainty, plus the 63-day forecast files (Baseline 3, Intermediate 3, 4 and 5)

**The question this notebook answers:** How wide should the forecast range be, is it honest (coverage), and how does skill change from 1 to 63 days?

> **How to use this starter (delete this box when you finish).** Steps 1 and 2 already work: run them
> first. Every other step says what to do and which course notebook to copy from. Replace each `# TODO`
> cell with your code, and each *italic* placeholder with your own words about **JPM's** results.
> Your full instructions are in your section of the team guide.

## Step 1: Set up

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tslab import config
from tslab.data import load_market_data
from tslab.team import build_features_jpm, make_dataset_jpm, FEATURE_GROUPS_JPM, ALL_FEATURES_JPM   # OUR features
from tslab.validation import walk_forward_splits, walk_forward_predict, describe_splits
from tslab.metrics import (compare_models, diebold_mariano, dm_verdict, directional_accuracy,
                           directional_accuracy_test, out_of_sample_r2, rmse, mae,
                           simple_backtest, backtest_summary)
from tslab.plots import plot_walk_forward_layout, plot_price_forecasts
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.api import SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
np.random.seed(config.SEED)
print("Target stock:", config.TARGET_TICKER)

## Step 2: The data and the walk-forward folds

Every notebook in our project uses these lines, so that every model is scored on the same days.

In [ ]:
frame = load_market_data()                                              # one row per JPM trading day
dataset_all = make_dataset_jpm(frame, horizon=1)                        # includes the last row (no target yet)
latest_row = dataset_all.iloc[[-1]]                                     # kept aside for a real forecast
dataset = dataset_all.dropna(subset=["target_return"])
splits = list(walk_forward_splits(len(dataset), config.INITIAL_TRAIN, config.TEST_SIZE))

print("Rows in the modelling table:", len(dataset))                     # expect 2789
print("Walk-forward folds         :", len(splits))                      # expect 29
print("First test day:", dataset.index[splits[0][1][0]].date(), "| last test day:", dataset.index[splits[-1][1][-1]].date())

## Step 3: Settings from Member 3

Load `outputs/ml_best_params.json` if Member 3 has pushed it; otherwise use Notebook 7's settings and say so.

In [ ]:
params_file = config.OUTPUT_DIR / "ml_best_params.json"
params = json.load(open(params_file)) if params_file.exists() else {"best_depth": 2, "best_rate": 0.03}
print("Gradient boosting settings:", params)

last_close, last_date = float(frame["Close"].iloc[-1]), frame.index[-1]
print(f"Last close {last_close:.2f} on {last_date.date()}")

## Step 4: 10% to 90% range and coverage

Walk-forward quantile gradient boosting for 0.10 and 0.90 (Notebook 4, Step 10). Report the coverage (target 80%), the coverage by year, and the average width in dollars. Is the model overconfident? Save `outputs/predictions_quantile.csv`.

In [ ]:
# TODO

## Step 5: Overlapping labels

Copy the gap demo from Notebook 7, Step 3 and explain why horizon h needs `gap = h - 1`.

In [ ]:
# TODO

## Step 6: Skill at 1, 5, 21 and 63 days

Copy Notebook 7, Step 4 with `make_dataset_jpm` / `ALL_FEATURES_JPM`. Add a DM test at each horizon (`diebold_mariano(..., horizon=h)`) and Ridge as a second method. Save `outputs/horizon_skill.csv`. Explain why skill changes.

In [ ]:
# TODO

## Step 7: The 63-day path on real trading days

Copy Notebook 7, Step 5, using these NYSE trading days (20 Nov 2026 to 23 Feb 2027).

In [ ]:
nyse_holidays = ["2026-11-26", "2026-12-25", "2027-01-01", "2027-01-18", "2027-02-15"]
trading_day = pd.offsets.CustomBusinessDay(holidays=nyse_holidays)
future_dates = pd.date_range(last_date + trading_day, periods=63, freq=trading_day)
print(future_dates[0].date(), "to", future_dates[-1].date())

# TODO: point, low and high price paths for each of the 63 days

## Step 8: Save forecast_basic.csv (exact column names)

The brief requires exactly `date, forecast, lower, upper`.

In [ ]:
# TODO: basic = pd.DataFrame({"date": future_dates, "forecast": ..., "lower": ..., "upper": ...})
# basic.to_csv(config.DATA_DIR / "forecast_basic.csv", index=False, date_format="%Y-%m-%d")
# assert list(pd.read_csv(config.DATA_DIR / "forecast_basic.csv").columns) == ["date", "forecast", "lower", "upper"]

## Step 9: Save forecast_intermediate.csv

63 rows: `date, naive, ses, arima, ridge, gradient_boosting, lower, upper`.

In [ ]:
# TODO

## Step 10: metrics_by_method.csv for Power BI

Score every column of `outputs/predictions_classical.csv` and `outputs/predictions_ml.csv` on the common days: `method, family, rmse_price, mae_price, r2_oos, directional_accuracy, dm_stat, dm_p, beats_naive`.

In [ ]:
# TODO

## Step 11: Chart

The last year of actual closes, the 63-day forecast and the shaded range (the same picture as Power BI).

In [ ]:
# TODO

## Conclusion

*TODO (your words, about JPM's numbers). If no model beats the naive forecast, say so plainly: it is a valid finding.*